In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from glob import glob

LAYER = 1

feats_a_name = glob(f"../../experiment_data/multiseed_matching/a{LAYER}/feats_*_a.csv")[0]
points_a_name = glob(f"../../experiment_data/multiseed_matching/a{LAYER}/points_*_a.csv")[0]
feats_b_name = glob(f"../../experiment_data/multiseed_matching/a{LAYER}/feats_*_b.csv")[0]
points_b_name = glob(f"../../experiment_data/multiseed_matching/a{LAYER}/points_*_b.csv")[0]
feats_c_name = glob(f"../../experiment_data/multiseed_matching/a{LAYER}/feats_*_c.csv")[0]
points_c_name = glob(f"../../experiment_data/multiseed_matching/a{LAYER}/points_*_c.csv")[0]

feats_a = pd.read_csv(feats_a_name)
points_a = pd.read_csv(points_a_name)
feats_b = pd.read_csv(feats_b_name)
points_b = pd.read_csv(points_b_name)
feats_c = pd.read_csv(feats_c_name)
points_c = pd.read_csv(points_c_name)

In [2]:
def make_merged_feats(points, feats):
    in_feat = pd.merge(points, feats, on=["Feature ID", "Simplification Threshold"], how="left")
    in_feat.drop(columns=["Simplification Threshold", "Homogeneity", "Majority Class", "Major Class Size", "Majority Class Coverage"], inplace=True)
    in_feat.rename(columns={"Feature ID": "fid", "Data Index": "idx", "Loss Start": "lfrom", "Loss End": "lto", "Feature Type": "typ", "Volume": "vol", "Persistence": "pers"}, inplace=True)
    return in_feat

def make_between(a, b):
    between = pd.merge(a, b, on=["idx"], how="inner", suffixes=("_f", "_t"))
    intersection = between.groupby(["fid_f", "fid_t"]).size().reset_index(name="int")
    props = intersection.merge(between, on=["fid_f", "fid_t"], how="left")
    props["union"] = props["vol_f"] + props["vol_t"] - props["int"]
    props["iou"] = props["int"] / props["union"]

    return props

feat_a = make_merged_feats(points_a, feats_a)
feat_b = make_merged_feats(points_b, feats_b)
feat_c = make_merged_feats(points_c, feats_c)

between_ab = make_between(feat_a, feat_b)
between_ac = make_between(feat_a, feat_c)
between_bc = make_between(feat_b, feat_c)
between_ba = make_between(feat_b, feat_a)
between_ca = make_between(feat_c, feat_a)
between_cb = make_between(feat_c, feat_b)

between_ab.head()

,fid_f,fid_t,int,idx,lfrom_f,lto_f,typ_f,pers_f,vol_f,lfrom_t,lto_t,typ_t,pers_t,vol_t,union,iou
0,0,7,6064,1,-0.0,0.204905,minima-saddle,0.01094,6391,-0.0,0.218537,minima-saddle,0.006018,6256,6583,0.921161
1,0,7,6064,6,-0.0,0.204905,minima-saddle,0.01094,6391,-0.0,0.218537,minima-saddle,0.006018,6256,6583,0.921161
2,0,7,6064,8,-0.0,0.204905,minima-saddle,0.01094,6391,-0.0,0.218537,minima-saddle,0.006018,6256,6583,0.921161
3,0,7,6064,17,-0.0,0.204905,minima-saddle,0.01094,6391,-0.0,0.218537,minima-saddle,0.006018,6256,6583,0.921161
4,0,7,6064,25,-0.0,0.204905,minima-saddle,0.01094,6391,-0.0,0.218537,minima-saddle,0.006018,6256,6583,0.921161


In [3]:
def get_best_iou_matching(between):
    best_iou = between.groupby("fid_f")["iou"].idxmax()
    best_matching = between.loc[best_iou]
    return best_matching

iou_ab = get_best_iou_matching(between_ab)
iou_ac = get_best_iou_matching(between_ac)

iou_ab_avg = iou_ab["iou"].mean()
iou_ac_avg = iou_ac["iou"].mean()

iou_ab_min_avg = iou_ab[iou_ab["typ_f"].str.contains("minima") & iou_ab["typ_t"].str.contains("minima")]["iou"].mean()
iou_ac_min_avg = iou_ac[iou_ac["typ_f"].str.contains("minima") & iou_ac["typ_t"].str.contains("minima")]["iou"].mean()

iou_ab_avg, iou_ac_avg, iou_ab_min_avg, iou_ac_min_avg

(np.float64(0.49333418476070673),
 np.float64(0.4977957146268687),
 np.float64(0.890874336748217),
 np.float64(0.8995876867806814))

In [6]:
iou_bc = get_best_iou_matching(between_cb)
iou_bc_avg = iou_bc["iou"].mean()
iou_bc_min_avg = iou_bc[iou_bc["typ_f"].str.contains("minima") & iou_bc["typ_t"].str.contains("minima")]["iou"].mean()

iou_bc_avg, iou_bc_min_avg

(np.float64(0.49165452548899213), np.float64(0.8869964183698682))

In [7]:
iou_bc = get_best_iou_matching(between_bc)
iou_bc_avg = iou_bc["iou"].mean()
iou_bc_min_avg = iou_bc[iou_bc["typ_f"].str.contains("minima") & iou_bc["typ_t"].str.contains("minima")]["iou"].mean()

iou_bc_avg, iou_bc_min_avg

(np.float64(0.49280802891587755), np.float64(0.8869964183698682))